# 🎙️ JORINOVA NEXUS — Voice Biometric Training (Colab GPU)

Builds an **accurate speaker voiceprint** per staff user with a GPU deep-speaker encoder
(**resemblyzer** GE2E d-vector, ~2% EER — far better than the on-device MFCC), calibrates the
verification threshold, and **exports** a `voiceprints.json` the backend loads for **voice login**
(speak → the system finds the matching user + their role, no typing).

**Runtime → Change runtime type → T4 GPU.** Run cells top→bottom.

> Two engines, one system:
> - **Capable machine** (this notebook / a server with `pip install resemblyzer`): deep d-vector — high accuracy.
> - **Weak machine / offline** (Render): the backend's pure-numpy MFCC path — already built, no deps.
>
> The embedding is math only — raw audio is never stored. Decision support; a human still approves enrolments.

## 1. Install (GPU)

In [ ]:
!pip -q install resemblyzer soundfile librosa scikit-learn
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — set Runtime>T4 GPU')

## 2. Provide the voice samples
Upload a **zip** whose folders are **usernames**, each with that user's `.wav`/`.webm`/`.mp3` clips
(3+ each, ~4s, normal working voice):
```
voices.zip
  └ jdoe/        s1.wav s2.wav s3.wav
  └ muadmin/     a1.wav a2.wav a3.wav
```
> No dataset yet? Run cell **2b** to record a couple of demo speakers in the browser.

In [ ]:
import os, zipfile, glob
from google.colab import files
os.makedirs('/content/voices', exist_ok=True)
up = files.upload()  # pick voices.zip
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('/content/voices')
root = '/content/voices'
# descend into a single wrapping folder if present
subs = [d for d in glob.glob(root+'/*') if os.path.isdir(d)]
if len(subs)==1 and not glob.glob(subs[0]+'/*.wav') and glob.glob(subs[0]+'/*/*'):
    root = subs[0]
users = sorted([os.path.basename(d) for d in glob.glob(root+'/*') if os.path.isdir(d)])
print('users:', users)
print('clips:', {u: len(glob.glob(f'{root}/{u}/*')) for u in users})

### 2b. (optional) record demo speakers in the browser

In [ ]:
# Records 4s clips via the browser mic and saves them under /content/voices/<username>/.
from IPython.display import Javascript, display
from google.colab import output
import base64, os
REC = '''
async function rec(sec){const s=await navigator.mediaDevices.getUserMedia({audio:true});
const m=new MediaRecorder(s);const c=[];m.ondataavailable=e=>c.push(e.data);m.start();
await new Promise(r=>setTimeout(r,sec*1000));m.stop();
await new Promise(r=>m.onstop=r);s.getTracks().forEach(t=>t.stop());
const b=new Blob(c);const buf=await b.arrayBuffer();
return btoa(String.fromCharCode(...new Uint8Array(buf)));}
'''
def record(username, n=3, sec=4):
    os.makedirs(f'/content/voices/{username}', exist_ok=True)
    display(Javascript(REC))
    for i in range(n):
        print(f'{username}: speak now ({i+1}/{n})…')
        data = output.eval_js(f'rec({sec})')
        open(f'/content/voices/{username}/clip{i}.webm','wb').write(base64.b64decode(data))
    print('saved', n, 'clips for', username)
# Example: record('jdoe'); record('muadmin')
print('call record("<username>") for each demo speaker, then re-run cell 2 detection below')

## 3. Encode every clip → d-vector (GPU)

In [ ]:
import numpy as np, glob, os
from resemblyzer import VoiceEncoder, preprocess_wav
enc = VoiceEncoder()  # uses GPU automatically if available

X, y = [], []            # per-clip embedding, username
for u in users:
    for f in glob.glob(f'{root}/{u}/*'):
        try:
            wav = preprocess_wav(f)
            X.append(enc.embed_utterance(wav)); y.append(u)
        except Exception as e:
            print('skip', f, e)
X = np.array(X); y = np.array(y)
print('embeddings:', X.shape, '| dim', X.shape[1] if len(X) else 0)

## 4. Per-user voiceprint + threshold calibration (EER)
Builds each user's mean voiceprint and finds the cosine threshold where false-accept ≈ false-reject.

In [ ]:
def cos(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))
prints = {u: X[y==u].mean(0) for u in users}      # centroid per user
prints = {u: v/ (np.linalg.norm(v)+1e-9) for u,v in prints.items()}

# genuine vs impostor scores (clip vs each user's centroid)
gen, imp = [], []
for emb, lab in zip(X, y):
    for u, c in prints.items():
        (gen if u==lab else imp).append(cos(emb,c))
gen, imp = np.array(gen), np.array(imp)
best_t, best_gap = 0.75, 9
for t in np.linspace(0.5,0.95,91):
    far = (imp>=t).mean() if len(imp) else 0      # false accept
    frr = (gen<t).mean()  if len(gen) else 0      # false reject
    if abs(far-frr) < best_gap: best_gap, best_t = abs(far-frr), float(t)
print(f'EER threshold ≈ {best_t:.3f}  (genuine mean {gen.mean():.3f}, impostor mean {imp.mean():.3f})')

## 5. Quick self-test (1:N identification — “voice → which user”)

In [ ]:
hits = 0
for emb, lab in zip(X, y):
    scores = {u: cos(emb,c) for u,c in prints.items()}
    who = max(scores, key=scores.get)
    hits += (who==lab and scores[who]>=best_t)
print(f'identification accuracy: {hits}/{len(y)} = {hits/max(1,len(y)):.1%} at threshold {best_t:.3f}')

## 6. Export → `voiceprints.json` (load into the backend)
Drop the file at **`backend/models/voice/voiceprints.json`**. The backend uses it for
**voice login** (1:N match → the matched username + role) when the `resemblyzer` engine is available.

In [ ]:
import json
out = {
    'engine': 'resemblyzer',
    'dim': int(X.shape[1]),
    'threshold': round(best_t, 3),
    'voiceprints': {u: prints[u].astype(float).round(6).tolist() for u in users},
}
open('/content/voiceprints.json','w').write(json.dumps(out))
print('users:', list(out['voiceprints'])); print('dim', out['dim'], 'threshold', out['threshold'])
from google.colab import files; files.download('/content/voiceprints.json')

## 7. Put it in the app
1. Move `voiceprints.json` → **`backend/models/voice/voiceprints.json`**, commit, push.
2. On a **capable server**, `pip install resemblyzer` → the backend's TIER-1 encoder + this file power
   accurate enrolment/verification and **voice login**.
3. On a **weak/offline machine**, nothing to install — the backend keeps working on the numpy-MFCC path
   (per-user 1:1 verify) automatically.

> Re-run this notebook whenever staff change to refresh the voiceprints. Raw audio is never exported — only the math.